# Linear Models Modeling

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
from pathlib import Path

pd.set_option("display.max_columns", 500)

sys.path.append(str(Path().cwd().parent.resolve()))

import preprocessing.features as features

builder = features.FeatureBuilder()

df = pd.read_csv(features.DATASET_PATH)

df_copy = builder.get_df(df)

df_copy.shape

/home/carl/notebooks/airbnb_prices_prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(74111, 27)

# Baseline Model. Linear Regression

This section establishes a baseline performance using a standard **Linear Regression** model without hyperparameters tuning. The goal is to obtain a reliable reference point before evaluating more advanced linear models such as Ridge, Lasso and ElasticNet.

For this baseline experiment, the dataset contains only engineered features, created during the feature engineering stage. Additional features sets, including **amenities** and **description sentence embeddings**, are intentionally excluded. Their predictive value will be evaluated separately in dedicated experiments after selecting the most suitable linear model.

The **zipcode** feature is also excluded from this baseline. Encoding this variable with One-Hot-Encoding would introduce more than 700 additional sparse features, substantially increasing the dimensionality of the dataset. Instead the model relies on other geographical features - **city**, **neighbourhood** and **distance_to_listing_center** - which are expected to capture the majority of spatial information, while keeping the feature space compact and interpratable.

Finally, the trained model will be evaluated on the test set using standard regression metrics to establish the baseline performance for subsequent experiments.

In [2]:
df_copy = builder.get_df(df)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 25), (74111,))

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train.shape, y_train.shape

((59288, 25), (59288,))

In [12]:
import preprocessing.linear_preprocessor as preprocessor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

cat_features = X_train.select_dtypes(include=['object', 'string']).columns
num_features = X_train.select_dtypes(include=['number']).columns

linear_preprocessor = preprocessor.create_linear_preprocessor(num_features=num_features, cat_features=cat_features)

linear_pipeline = Pipeline([
    ("preprocessor", linear_preprocessor),
    ("model", LinearRegression())
])

linear_pipeline.fit(X_train, y_train)
y_pred = linear_pipeline.predict(X_test)

In [13]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

print(f"MAE : {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.4f}")
print(f"R²  : {r2_score(y_test, y_pred):.4f}")

MAE : 0.3083
RMSE: 0.4191
R²  : 0.6581
